# Synthetic SFT Data Generator

Uses **qwen/qwen3.5-flash-02-23** via OpenRouter to generate ~3000 synthetic SFT records.

**Two fine-tuned datasets**:
- `synthetic_justifier_sft.jsonl` — is_relevant true+false (Justifier model)
- `synthetic_enricher_sft.jsonl` — is_relevant=true only (Enricher model)

**Temperature**: Keyword/Article/Enricher=0.7, Justifier=**0.3**
**Seeds**: Both `justifier_sft.jsonl` (260) + `enricher_sft.jsonl` (100) = ~360 unique seeds

In [ ]:
# Setup
import os, json, re, time
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('.env'))
from openai import OpenAI
import sys

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from services.llm.prompts import (
    JUSTIFIER_SYSTEM, ENRICHER_SYSTEM,
    build_justifier_prompt, build_enricher_prompt,
    build_messages,
)

OPENROUTER_API_KEY    = os.environ['OPENROUTER_API_KEY']
OPENROUTER_BASE       = 'https://openrouter.ai/api/v1'
MODEL                 = 'qwen/qwen3.5-flash-02-23'
KEYWORDS_PER_CATEGORY = 26
ARTICLES_PER_KEYWORD  = 3

TEMP_KEYWORD   = 0.7
TEMP_ARTICLE   = 0.7
TEMP_JUSTIFIER = 0.3
TEMP_ENRICHER  = 0.7

OUTPUT_DIR    = PROJECT_ROOT / 'data/sft/keyword-scraper-2'
JUSTIFIER_OUT = OUTPUT_DIR / 'synthetic_justifier_sft.jsonl'
ENRICHER_OUT  = OUTPUT_DIR / 'synthetic_enricher_sft.jsonl'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE)

def chat(model, messages, temperature=0.7, *, max_retries=3):
    for attempt in range(max_retries):
        try:
            r = client.chat.completions.create(model=model, messages=messages, temperature=temperature)
            return r.choices[0].message.content.strip()
        except Exception as e:
            wait = (attempt+1)*5 + (attempt*2)
            print(f'  [retry {attempt+1}] {e} -- waiting {wait}s')
            time.sleep(wait)
    raise RuntimeError('chat failed')

CATEGORIES = {
    'integritas_penegakan_hukum': (
        'kasus korupsi, suap, gratifikasi, penegakan hukum oleh KPK, Polri, kejaksaan, '
        'serta isu integritas, transparansi, dan etika pejabat publik'
    ),
    'kebijakan_layanan_publik': (
        'kebijakan pemerintah pusat dan daerah, perpres, permen, undang-undang, '
        'serta efektivitas dan kualitas penyediaan layanan publik bagi masyarakat'
    ),
    'kinerja_pemerintah': (
        'evaluasi kinerja kabinet, kementerian, pemerintah daerah, DPR/DPRD, '
        'serta capaian program, birokrasi, dan tata kelola pemerintahan secara umum'
    ),
    'keamanan_siber_ketertiban_digital': (
        'keamanan siber, perlindungan data pribadi, penanganan peretasan, '
        'literasi digital, ketertiban ruang siber, dan regulasi elektronik'
    ),
    'kesejahteraan_bantuan_sosial': (
        'program perlindungan sosial, pengentasan kemiskinan, subsidi, '
        'bantuan sosial (bansos), PKH, dan jaminan sosial masyarakat'
    ),
    'infrastruktur_layanan_transportasi': (
        'proyek pembangunan fisik seperti jalan tol, jembatan, bandara, pelabuhan, '
        'serta ketersediaan, kualitas, dan tata kelola layanan transportasi publik'
    ),
    'ekonomi_digital': (
        'perkembangan industri teknologi, startup, e-commerce, fintech, '
        'digitalisasi UMKM, dan inovasi perekonomian berbasis digital'
    ),
    'ketenagakerjaan': (
        'ketersediaan lapangan kerja, tingkat pengangguran, isu buruh, '
        'regulasi upah minimum, hak pekerja, dan dinamika hubungan industrial'
    ),
    'layanan_kesehatan': (
        'akses dan kualitas fasilitas kesehatan, BPJS Kesehatan, JKN, '
        'penanganan wabah, gizi buruk, dan kebijakan kesehatan masyarakat'
    ),
    'pendidikan_pengembangan_sdm': (
        'sistem pendidikan dasar hingga tinggi, kurikulum, beasiswa pemerintah, '
        'kesejahteraan guru, dan program peningkatan kapasitas sumber daya manusia'
    ),
    'krisis_sosial_kebencanaan': (
        'mitigasi dan penanganan darurat bencana alam, perubahan iklim, '
        'kerusakan lingkungan, krisis sosial, dan manajemen tanggap darurat'
    ),
}

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'Files        : justifier={(OUTPUT_DIR/"justifier_sft.jsonl").exists()}, enricher={(OUTPUT_DIR/"enricher_sft.jsonl").exists()}')
print(f'Categories   : {len(CATEGORIES)}  (justifier_temp={TEMP_JUSTIFIER})')


In [ ]:
# Load seed keywords from BOTH JSONL files (deduplicated)
def load_keywords(path):
    kw = []
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            uc = next((m['content'] for m in rec['messages'] if m['role'] == 'user'), '')
            m = re.search(r'Keyword (?:trending|utama): (.+?)(?:\n|$)', uc)
            if m:
                kw.append(m.group(1).strip())
    return kw

seeds = set()
for fp in [OUTPUT_DIR / 'justifier_sft.jsonl', OUTPUT_DIR / 'enricher_sft.jsonl']:
    for k in load_keywords(fp):
        seeds.add(k.lower().strip())

real_keyword_texts = sorted(seeds, key=str.lower)
print(f'Unique seed keywords: {len(real_keyword_texts)}')
print(f'  Sample: {real_keyword_texts[:5]}')


In [ ]:
# Step 1: Generate category-specific keyword variants (temp=0.7)
KW_PROMPT = (
    'Kamu adalah generator kata kunci tren untuk sistem monitoring isu pemerintahan Indonesia.\n\n'
    'Kategori: {cat_name}\n'
    'Deskripsi: {cat_desc}\n\n'
    'Buat tepat {count} kata kunci tren Indonesia yang relevan dengan kategori ini.\n'
    '- Berbahasa Indonesia (bisa campur Inggris jika umum)\n'
    '- Spesifik, bukan generik (misal: pemadatan kebakaran Jakarta bukan berita jakarta)\n'
    '- Mencerminkan topik yang sedang tren atau penting di Indonesia\n'
    '- Variatif: campur antara nama lembaga, kebijakan, lokasi, dan isu spesifik\n\n'
    'Seed inspiration: {seed}\n\n'
    'Respond HANYA dengan JSON array string.\n'
    "Format: ['kata kunci 1', 'kata kunci 2', ...]"
)

def gen_keywords(cat_key, cat_desc, count, seed):
    prompt = KW_PROMPT.format(cat_name=cat_key, cat_desc=cat_desc, count=count, seed=seed)
    try:
        resp = chat(MODEL, [{'role': 'user', 'content': prompt}], temperature=TEMP_KEYWORD)
        m = re.search(r'\[.*\]', resp, re.DOTALL)
        if m:
            return [k.strip() for k in json.loads(m.group()) if k.strip()]
        print(f'  [warn] {resp[:80]}')
    except Exception as e:
        print(f'  [error] {e}')
    return []

all_synthetic = []
cat_keys = list(CATEGORIES.keys())
for i, (ck, cd) in enumerate(CATEGORIES.items()):
    seed = real_keyword_texts[i % len(real_keyword_texts)]
    half = KEYWORDS_PER_CATEGORY // 2
    from_seed    = gen_keywords(ck, cd, half, seed)
    from_scratch = gen_keywords(ck, cd, KEYWORDS_PER_CATEGORY - half, '(tidak ada seed)')
    combined = from_seed + from_scratch
    seen, unique = set(), []
    for k in combined:
        kl = k.lower()
        if kl not in seen:
            seen.add(kl)
            unique.append(k)
    for kw in unique:
        all_synthetic.append({'keyword': kw, 'category': ck})
    print(f'  [{i+1:02d}/{len(cat_keys)}] {ck}: {len(unique)} keywords')
    time.sleep(0.5)
print(f'\nTotal synthetic keywords: {len(all_synthetic)}')


In [ ]:
# Step 2: Generate synthetic articles (temp=0.7)
ART_PROMPT = (
    'Kamu adalah generator artikel sintetis untuk data training model AI.\n\n'
    'Buat {count} artikel pendek (gaya berita Indonesia) untuk kata kunci: {keyword}\n\n'
    'Setiap artikel WAJIB memiliki:\n'
    '- title: judul spesifik, maksimal 12 kata\n'
    '- body: 2-3 paragraf, 150-300 kata, gaya berita Indonesia formal\n'
    '- summary: ringkasan 1-2 kalimat, maksimal 50 kata\n\n'
    'Artikel harus:\n'
    '- Berbahasa Indonesia formal\n'
    '- Mengandung nama lembaga pemerintah, kebijakan, atau tokoh resmi Indonesia\n'
    '- Netral dan faktual\n\n'
    'Respond HANYA dengan JSON array.\n'
    "Format: [{'title': '...', 'body': '...', 'summary': '...'}, ...]"
)

class SArt:
    def __init__(self, title, body, summary):
        self.title, self.body, self.summary = title, body, summary

def parse_arts(resp):
    m = re.search(r'\[.*\]', resp, re.DOTALL)
    if m:
        try:
            arts = json.loads(m.group())
            if isinstance(arts, list):
                return [SArt(a.get('title',''), a.get('body',''), a.get('summary',''))
                         for a in arts if a.get('title') and a.get('body')]
        except json.JSONDecodeError:
            pass
    print(f'  [warn] {resp[:100]}')
    return []

def gen_arts(keyword, count=ARTICLES_PER_KEYWORD):
    try:
        resp = chat(MODEL, [{'role': 'user', 'content': ART_PROMPT.format(keyword=keyword, count=count)}],
                   temperature=TEMP_ARTICLE)
        return parse_arts(resp)
    except Exception as e:
        print(f'  [error] {keyword[:30]}...: {e}')
        return []

print(f'Generating {ARTICLES_PER_KEYWORD} articles per keyword...')
print(f'Total: {len(all_synthetic)} keywords = {len(all_synthetic)*ARTICLES_PER_KEYWORD} articles\n')
for idx, item in enumerate(all_synthetic):
    item['articles'] = gen_arts(item['keyword'])
    if (idx+1) % 20 == 0:
        print(f'  [{idx+1}/{len(all_synthetic)}] processed')
    time.sleep(0.3)
with_articles = [it for it in all_synthetic if it['articles']]
print(f'\nKeywords with articles: {len(with_articles)}/{len(all_synthetic)}')


In [ ]:
# Step 3: Run Justifier on ALL keywords (temp=0.3)
def build_ctx(arts):
    parts = []
    for i, a in enumerate(arts, 1):
        parts.append(f'[Artikel {i}] {a.title}\n{a.body}')
    return '\n\n'.join(parts)

def run_justifier(keyword, arts):
    ctx = build_ctx(arts)
    msgs = build_messages(JUSTIFIER_SYSTEM, build_justifier_prompt(keyword, ctx))
    try:
        resp = chat(MODEL, msgs, temperature=TEMP_JUSTIFIER)
        clean = resp.strip()
        for p in ('```json', '```'):
            if clean.startswith(p): clean = clean[len(p):]
        if clean.endswith('```'): clean = clean[:-3]
        data = json.loads(clean.strip())
        return {'is_relevant': bool(data.get('is_relevant')),
                'justification': str(data.get('justification', ''))}
    except Exception as e:
        print(f'  [justifier error] {keyword[:25]}...: {e}')
        return {'is_relevant': False, 'justification': 'Parse error'}

print(f'Running Justifier (temp={TEMP_JUSTIFIER}) on {len(with_articles)} keywords...\n')
for idx, item in enumerate(with_articles):
    item['just_result'] = run_justifier(item['keyword'], item['articles'])
    if (idx+1) % 20 == 0:
        print(f'  [{idx+1}/{len(with_articles)}] processed')
    time.sleep(0.3)
relevant = [it for it in with_articles if it['just_result']['is_relevant']]
print(f'\nJustifier done: {len(relevant)}/{len(with_articles)} relevant')


In [ ]:
# Step 4: Run Enricher on is_relevant=true only (temp=0.7)
def run_enricher(keyword, arts):
    ctx = build_ctx(arts)
    msgs = build_messages(ENRICHER_SYSTEM, build_enricher_prompt(keyword, ctx))
    try:
        resp = chat(MODEL, msgs, temperature=TEMP_ENRICHER)
        clean = resp.strip()
        for p in ('```json', '```'):
            if clean.startswith(p): clean = clean[len(p):]
        if clean.endswith('```'): clean = clean[:-3]
        data = json.loads(clean.strip())
        kws = data.get('expanded_keywords', [])
        return {'expanded_keywords': kws if isinstance(kws, list) else []}
    except Exception as e:
        print(f'  [enricher error] {keyword[:25]}...: {e}')
        return {'expanded_keywords': [keyword]}

print(f'Running Enricher (temp={TEMP_ENRICHER}) on {len(relevant)} relevant keywords...\n')
for idx, item in enumerate(relevant):
    item['enrich_result'] = run_enricher(item['keyword'], item['articles'])
    if (idx+1) % 20 == 0:
        print(f'  [{idx+1}/{len(relevant)}] processed')
    time.sleep(0.3)
enriched = [it for it in relevant if it.get('enrich_result')]
print(f'\nEnricher done: {len(enriched)} enriched')


In [ ]:
# Step 5: Write two separate JSONL files
for path in [JUSTIFIER_OUT, ENRICHER_OUT]:
    if path.exists():
        path.unlink()

just_count, enr_count = 0, 0
for item in with_articles:
    kw, arts = item['keyword'], item['articles']
    just = item['just_result']
    enrich = item.get('enrich_result')
    ctx = build_ctx(arts)
    # Justifier record (ALL keywords -- needs true+false to learn boundary)
    msgs = build_messages(JUSTIFIER_SYSTEM, build_justifier_prompt(kw, ctx))
    ac = json.dumps({'is_relevant': just['is_relevant'],
                    'justification': just['justification']}, ensure_ascii=False)
    with open(JUSTIFIER_OUT, 'a', encoding='utf-8') as f:
        f.write(json.dumps({'messages': msgs + [{'role': 'assistant', 'content': ac}]}, ensure_ascii=False) + '\n')
    just_count += 1
    # Enricher record (is_relevant=true only -- matches production distribution)
    if just['is_relevant'] and enrich:
        msgs = build_messages(ENRICHER_SYSTEM, build_enricher_prompt(kw, ctx))
        ac = json.dumps({'expanded_keywords': enrich['expanded_keywords']}, ensure_ascii=False)
        with open(ENRICHER_OUT, 'a', encoding='utf-8') as f:
            f.write(json.dumps({'messages': msgs + [{'role': 'assistant', 'content': ac}]}, ensure_ascii=False) + '\n')
        enr_count += 1

print(f'Export complete!')
print(f'  synthetic_justifier_sft.jsonl: {just_count} records')
print(f'  synthetic_enricher_sft.jsonl : {enr_count} records')


In [ ]:
# Preview & Stats
import subprocess
print('=' * 60)
print('JUSTIFIER SAMPLE:')
print('=' * 60)
with open(JUSTIFIER_OUT) as f:
    d = json.loads(f.readline())
    for m in d['messages']:
        print(f'\n[{m["role"].upper()}]\n{m["content"][:300]}')
print('\n' + '=' * 60)
print('ENRICHER SAMPLE:')
print('=' * 60)
with open(ENRICHER_OUT) as f:
    d = json.loads(f.readline())
    for m in d['messages']:
        print(f'\n[{m["role"].upper()}]\n{m["content"][:300]}')
print('\n' + '=' * 60)
print('FILE STATS:')
print('=' * 60)
r = subprocess.run(['wc', '-l', str(JUSTIFIER_OUT), str(ENRICHER_OUT)], capture_output=True, text=True)
print(r.stdout)
